In [108]:
import pandas as pd

df = pd.read_csv("../data/googleplaystore.csv")
df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,7-Jan-18,1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,15-Jan-18,2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,1-Aug-18,1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,8-Jun-18,Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,20-Jun-18,1.1,4.4 and up


In [109]:
df.shape

(10841, 13)

In [110]:
df.columns

Index(['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type',
       'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver',
       'Android Ver'],
      dtype='str')

In [111]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  str    
 1   Category        10841 non-null  str    
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  str    
 4   Size            10841 non-null  str    
 5   Installs        10841 non-null  str    
 6   Type            10840 non-null  str    
 7   Price           10841 non-null  str    
 8   Content Rating  10840 non-null  str    
 9   Genres          10841 non-null  str    
 10  Last Updated    10841 non-null  str    
 11  Current Ver     10833 non-null  str    
 12  Android Ver     10838 non-null  str    
dtypes: float64(1), str(12)
memory usage: 1.1 MB


In [112]:
df.describe()

,Rating
count,9367.000000
mean,4.193338
std,0.537431
min,1.000000
25%,4.000000
50%,4.300000
75%,4.500000
max,19.000000


In [113]:
df.duplicated().sum()

np.int64(483)

In [114]:
# Remove duplicates
df = df.drop_duplicates(subset=["App"], keep="first")
df.duplicated().sum()

np.int64(0)

In [116]:
df[df["Rating"] > 5]

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
10472,Life Made WI-Fi Touchscreen Photo Frame,1.9,19.0,3.0M,"1,000+",Free,0,Everyone,NaN,11-Feb-18,1.0.19,4.0 and up,NaN


Notice that the columns are misaligned. Either remove or shift right depends on usage.
Should be removed since isolated case.

In [117]:
df = df.drop(index=10472)
df["Rating"].describe()

count    8196.000000
mean        4.173243
std         0.536625
min         1.000000
25%         4.000000
50%         4.300000
75%         4.500000
max         5.000000
Name: Rating, dtype: float64

In [118]:
df["Rating"].isna().sum()

np.int64(1463)

1465/10841=13.5%. This is a significant number. Drop since analyzing ratings.

In [119]:
df = df.dropna(subset=["Rating"])

Since we are analyzing which app category is rated highest by users, we can use groupby method


In [120]:
df.groupby("Category")["Rating"].mean().sort_values(ascending=False).head(10)

Category
EVENTS                 4.435556
EDUCATION              4.364407
ART_AND_DESIGN         4.357377
BOOKS_AND_REFERENCE    4.344970
PERSONALIZATION        4.332215
PARENTING              4.300000
BEAUTY                 4.278571
GAME                   4.247368
SOCIAL                 4.247291
WEATHER                4.243056
Name: Rating, dtype: float64

This method only shows highest rating categories but doesn't consider about the volume of each category, which can lead to misinterpreting. We can use 'agg()' instead of 'mean()' to interpret more efficiency.

In [121]:
df.groupby("Category")["Rating"].agg(["mean", "count"]).sort_values("mean", ascending=False).head(10)

,mean,count
Category,,
EVENTS,4.435556,45
EDUCATION,4.364407,118
ART_AND_DESIGN,4.357377,61
BOOKS_AND_REFERENCE,4.344970,169
PERSONALIZATION,4.332215,298
PARENTING,4.300000,50
BEAUTY,4.278571,42
GAME,4.247368,912
SOCIAL,4.247291,203


Filter catogories that have more than 100 apps for better analysis

In [122]:
df.groupby("Category")["Rating"].agg(["mean", "count"]).query("count >= 100").sort_values("mean", ascending=False).head(10)

,mean,count
Category,,
EDUCATION,4.364407,118
BOOKS_AND_REFERENCE,4.344970,169
PERSONALIZATION,4.332215,298
GAME,4.247368,912
SOCIAL,4.247291,203
HEALTH_AND_FITNESS,4.243033,244
SHOPPING,4.230000,180
SPORTS,4.216154,260
PRODUCTIVITY,4.183389,301


Depends business goal, if we're going with the scale strategy, we focus in the GAME category, where the volume is high but also mean: oversaturated market, hard to compete, lower profit margins. Or we can go with the high-satisfaction strategy, which is EDUCATION: high satisfaction, less saturation, niche growth opportunity.

Now we check market size


In [123]:
df["Installs"].dtype

<StringDtype(storage='python', na_value=nan)>

Since it's string type, we need to clean the data to integer to calculate the market size. 

In [124]:
df["Installs"] = df["Installs"].astype(str).str.replace("+", "", regex=False).str.replace(",", "", regex=False)
df["Installs"] = pd.to_numeric(df["Installs"], errors="coerce")
df["Installs"].dtype

dtype('int64')

In [125]:
df.groupby("Category")["Installs"].sum().sort_values(ascending=False).head(10)

Category
GAME                  13878762717
COMMUNICATION         11038241530
TOOLS                  7999724500
PRODUCTIVITY           5793070180
SOCIAL                 5487841475
PHOTOGRAPHY            4649143130
FAMILY                 4427479590
VIDEO_PLAYERS          3926797200
TRAVEL_AND_LOCAL       2894859300
NEWS_AND_MAGAZINES     2369110650
Name: Installs, dtype: int64

We see that game category dominates so this confirms that the category is in huge demand with massive user base and high engagement market. So for maximum reach we can go for GAME and COMMUNICATION; EDUCATION for highest satisfaction. Or we can play safe going for PRODUCTIVITY or SOCIAL.

In [126]:
df.groupby("Category")["Installs"] \
    .agg(["sum", "mean", "count"]) \
    .sort_values("mean", ascending=False) \
    .head(10)

,sum,mean,count
Category,,,
COMMUNICATION,11038241530,4.311813e+07,256
SOCIAL,5487841475,2.703370e+07,203
VIDEO_PLAYERS,3926797200,2.653241e+07,148
ENTERTAINMENT,2113660000,2.072216e+07,102
PRODUCTIVITY,5793070180,1.924608e+07,301
PHOTOGRAPHY,4649143130,1.767735e+07,263
TRAVEL_AND_LOCAL,2894859300,1.548053e+07,187
GAME,13878762717,1.521794e+07,912
NEWS_AND_MAGAZINES,2369110650,1.161329e+07,204


From this table we can see that COMMUNICATION standing at #1 while GAME only at #6. A COMMUNICATION app performs 2.3x better than a GAME app, so possibly has stronger network effects.

In [127]:
df.groupby("Category")["Installs"] \
  .agg(["mean", "median", "max", "count"]) \
  .sort_values("mean", ascending=False) \
  .head(10)

,mean,median,max,count
Category,,,,
COMMUNICATION,4.311813e+07,1000000.0,1000000000,256
SOCIAL,2.703370e+07,500000.0,1000000000,203
VIDEO_PLAYERS,2.653241e+07,1000000.0,1000000000,148
ENTERTAINMENT,2.072216e+07,1000000.0,1000000000,102
PRODUCTIVITY,1.924608e+07,500000.0,1000000000,301
PHOTOGRAPHY,1.767735e+07,1000000.0,1000000000,263
TRAVEL_AND_LOCAL,1.548053e+07,500000.0,1000000000,187
GAME,1.521794e+07,1000000.0,1000000000,912
NEWS_AND_MAGAZINES,1.161329e+07,100000.0,1000000000,204


Since COMMUNICATION and GAME have mean > median, it's very difficult for us to break in as a startup and expect to be performed because of right-skewed distribution(Winner-takes-all market).
From earlier ratings, EDUCATION would be a better choice since it has high satisfaction while the size is smaller.

In [128]:
df.groupby("Category")["Installs"] \
  .agg(["mean", "median", "std"]) \
  .sort_values("std", ascending=False) \
  .head(10)

,mean,median,std
Category,,,
COMMUNICATION,4.311813e+07,1000000.0,1.651671e+08
SOCIAL,2.703370e+07,500000.0,1.306430e+08
VIDEO_PLAYERS,2.653241e+07,1000000.0,1.232661e+08
TRAVEL_AND_LOCAL,1.548053e+07,500000.0,1.036479e+08
ENTERTAINMENT,2.072216e+07,1000000.0,1.005757e+08
NEWS_AND_MAGAZINES,1.161329e+07,100000.0,8.530908e+07
PRODUCTIVITY,1.924608e+07,500000.0,8.347238e+07
BOOKS_AND_REFERENCE,9.856755e+06,100000.0,7.813792e+07
PHOTOGRAPHY,1.767735e+07,1000000.0,6.651404e+07


In [129]:
comm_df = df[df["Category"] == "COMMUNICATION"]
top3 = comm_df.nlargest(3, "Installs")[["App", "Installs"]]
bot3 = comm_df.nsmallest(3, "Installs")[["App", "Installs"]]
top3, bot3

(                                          App    Installs
 335  Messenger – Text and Video Chat for Free  1000000000
 336                        WhatsApp Messenger  1000000000
 338              Google Chrome: Fast & Secure  1000000000,
                                     App  Installs
 6046  Best Browser BD social networking        10
 8497                         DK Browser        10
 9455                       EJ messenger        10)

In [130]:
df[df["App"] == "Google Chrome: Fast & Secure"]

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
338,Google Chrome: Fast & Secure,COMMUNICATION,4.3,9642995,Varies with device,1000000000,Free,0,Everyone,Communication,1-Aug-18,Varies with device,Varies with device
